# Loading and Accessing Data

This notebook demonstrates how to load datasets from manifests or archives and access different types of protein data. We'll explore the Pythonic API for working with sequences, structures, assays, and MSAs.

## Loading Datasets

There are two main ways to load a PG2 dataset:
1. From a manifest file (TOML)
2. From a dataset archive (ZIP)

Let's start by importing the necessary modules:

In [1]:
import dataclasses
from pathlib import Path
from proteingym.base import Dataset, Manifest

# Set up paths
manifest_path = Path("../../example_data/neime_2019.toml")

/Users/ethan/miniconda3/envs/pgbase/lib/python3.14/site-packages/pydantic/_internal/_generate_schema.py:663: ArbitraryTypeWarning: <built-in function array> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warnings.warn(


### Method 1: Loading from Manifest

In [2]:
# Load manifest first
try:
    manifest = Manifest.from_path(manifest_path)
    print(f"Loaded manifest: {manifest.name}")

    # Create dataset from manifest
    dataset = Dataset.from_manifest(manifest)
    print(f"\nDataset created successfully!")
    print(f"Dataset name: {dataset.name}")

except Exception as e:
    print(f"Error loading from manifest: {e}")
    print("This might be due to missing data files or incorrect paths.")

Loaded manifest: NEIME_2019

Dataset created successfully!
Dataset name: NEIME_2019


In [3]:
dataset.assays[0].records[0:2]

[(Sequence(
  	name='ITLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQKSAVTEYYLNHGEWPGDNSSAGVATSADIKGKYVQSVTVANGVITAQMASSNVNNEIKSKKLSLWAKRQNGSVKWFCGQPVTRTTATATDVAAANGKTDDKINTKHLPSTCRDDSSAS',
  	description: None,
  	type: standard_sequence,
  	alphabet: AA,
  	value: ITLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQKSAVTEYYLNHGEWPGD...,
  ),
  -3.5980000000000003,
  0,
  'valid'),
 (Sequence(
  	name='LTLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQKSAVTEYYLNHGEWPGDNSSAGVATSADIKGKYVQSVTVANGVITAQMASSNVNNEIKSKKLSLWAKRQNGSVKWFCGQPVTRTTATATDVAAANGKTDDKINTKHLPSTCRDDSSAS',
  	description: None,
  	type: standard_sequence,
  	alphabet: AA,
  	value: LTLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQKSAVTEYYLNHGEWPGD...,
  ),
  -0.6779999999999999,
  0,
  'train')]

In [4]:
# Dump the data into .pgdata file.
dataset.dump(path="../../example_data/")

PosixPath('../../example_data/NEIME_2019.pgdata')

### Method 2: Loading from Archive

If you have a dataset archive (created in the previous notebook), you can load it directly:

In [5]:
# Look for existing archives
archive_path = "../../example_data/NEIME_2019.pgdata"

try:
    dataset = Dataset.from_path(archive_path)
except Exception as e:
    print(f"Error loading from archive: {e}")
    print(f"Did you create an archive in the previous tutorial?")
    raise e

# The `Dataset` is a Pydantic BaseModel that validates the data upon loading.
# A `BaseModel`` has a `.model_fields` attribute returning the fields of the class.
Dataset.model_fields

{'name': FieldInfo(annotation=str, required=True, description='The name of the dataset.'),
 'description': FieldInfo(annotation=Union[str, NoneType], required=False, default=None, description='A brief description of the dataset.'),
 'reference_sequence_name': FieldInfo(annotation=Union[str, NoneType], required=False, default=None, description='Name of the sequence that is to be considered the reference for this dataset.\n\nUseful for e.g. zero-shot models that compare likelihood for a token from a\nreference with the token for a variant.'),
 'assay_variables': FieldInfo(annotation=list[Field], required=False, default_factory=list, description='The list of assay variables relevant to the dataset.'),
 'assay_targets': FieldInfo(annotation=list[Field], required=False, default_factory=list, description='The list of assay targets relevant to the dataset.'),
 'assays': FieldInfo(annotation=list[Assay], required=False, default_factory=list, description='The assays present in the dataset.'),
 

In [6]:
# The `Dataset`` attributes are Python-native dataclasses
dataclasses.fields(dataset.assays[0])

(Field(name='name',type=<class 'str'>,default=<dataclasses._MISSING_TYPE object at 0x101ecdfd0>,default_factory=<dataclasses._MISSING_TYPE object at 0x101ecdfd0>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=True,doc=None,_field_type=_FIELD),
 Field(name='description',type=str | None,default=None,default_factory=<dataclasses._MISSING_TYPE object at 0x101ecdfd0>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=True,doc=None,_field_type=_FIELD),
 Field(name='fields',type=list[proteingym.base.assay.Field],default=<dataclasses._MISSING_TYPE object at 0x101ecdfd0>,default_factory=<function AssayRaw.<lambda> at 0x1088f6610>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=True,doc=None,_field_type=_FIELD),
 Field(name='records',type=list[tuple[proteingym.base.sequence.Sequence | str | int | float | bool | None, ...]],default=<dataclasses._MISSING_TYPE object at 0x101ecdfd0>,default_factory=<class 'list'>,i

## Exploring Dataset Structure

Let's examine what's in our dataset:

In [7]:
print(f"Dataset: {dataset.name}")
print(f"Description: {dataset.description}")
print("\nDataset contents:")
print(f"  - Sequences: {len(dataset.sequences)}")
print(f"  - Structures: {len(dataset.structures)}")
print(f"  - MSAs: {len(dataset.msas)}")
print(f"  - Assays: {len(dataset.assays)}")
print(f"  - Assay variables: {len(dataset.assay_variables)}")

Dataset: NEIME_2019
Description: The NEIME Kennouche 2019 (UniProt id: A0A1I9GEU1) datase

Dataset contents:
  - Sequences: 1
  - Structures: 1
  - MSAs: 1
  - Assays: 1
  - Assay variables: 2


## Accessing Assays

In [8]:
# Access the assays
assays = dataset.assays

# Extract an specific assay
my_assay = assays[0]

# We can get a summary of the data encoded in this assay:
for field in dataclasses.fields(my_assay):
    print(f"Found a field:\n{field}\n------------")

Found a field:
Field(name='name',type=<class 'str'>,default=<dataclasses._MISSING_TYPE object at 0x101ecdfd0>,default_factory=<dataclasses._MISSING_TYPE object at 0x101ecdfd0>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=True,doc=None,_field_type=_FIELD)
------------
Found a field:
Field(name='description',type=str | None,default=None,default_factory=<dataclasses._MISSING_TYPE object at 0x101ecdfd0>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=True,doc=None,_field_type=_FIELD)
------------
Found a field:
Field(name='fields',type=list[proteingym.base.assay.Field],default=<dataclasses._MISSING_TYPE object at 0x101ecdfd0>,default_factory=<function AssayRaw.<lambda> at 0x1088f6610>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=True,doc=None,_field_type=_FIELD)
------------
Found a field:
Field(name='records',type=list[tuple[proteingym.base.sequence.Sequence | str | int | float | bool | None, ...]

In [9]:
# Access specific attributes such as name
name = my_assay.name
print(f"Assay name: {name}")

# Or extract the records
records = my_assay.records
print(f"{name} contains {len(records)} records")
print(f"record 1: {records[0][0]}... \n with value {records[0][1]}")

Assay name: NEIME_2019
NEIME_2019 contains 922 records
record 1: Sequence(
	name='ITLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQKSAVTEYYLNHGEWPGDNSSAGVATSADIKGKYVQSVTVANGVITAQMASSNVNNEIKSKKLSLWAKRQNGSVKWFCGQPVTRTTATATDVAAANGKTDDKINTKHLPSTCRDDSSAS',
	description: None,
	type: standard_sequence,
	alphabet: AA,
	value: ITLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQKSAVTEYYLNHGEWPGD...,
)... 
 with value -3.5980000000000003


In [10]:
dataset_assays_df = dataset.to_df()
print(dataset_assays_df)

shape: (922, 5)
┌─────────────────────────────────┬─────┬─────┬───────────┬───────────────┐
│ sequence                        ┆ PH  ┆ T   ┆ DMS Score ┆ DMS Score Bin │
│ ---                             ┆ --- ┆ --- ┆ ---       ┆ ---           │
│ str                             ┆ i32 ┆ i32 ┆ f64       ┆ i64           │
╞═════════════════════════════════╪═════╪═════╪═══════════╪═══════════════╡
│ FALIELMIVIAIVGILAAVALPAYQDYTAR… ┆ 7   ┆ 37  ┆ 0.786     ┆ 1             │
│ FILIELMIVIAIVGILAAVALPAYQDYTAR… ┆ 7   ┆ 37  ┆ 1.149     ┆ 1             │
│ FNLIELMIVIAIVGILAAVALPAYQDYTAR… ┆ 7   ┆ 37  ┆ 1.111     ┆ 1             │
│ FPLIELMIVIAIVGILAAVALPAYQDYTAR… ┆ 7   ┆ 37  ┆ -0.107    ┆ 0             │
│ FSLIELMIVIAIVGILAAVALPAYQDYTAR… ┆ 7   ┆ 37  ┆ 0.217     ┆ 1             │
│ …                               ┆ …   ┆ …   ┆ …         ┆ …             │
│ ITLIELMIVIAIVGILAAVALPAYQDYTAR… ┆ 7   ┆ 37  ┆ -3.598    ┆ 0             │
│ LTLIELMIVIAIVGILAAVALPAYQDYTAR… ┆ 7   ┆ 37  ┆ -0.678    ┆ 0           

## Accessing Assay Variables

Assay variables describe the experimental setup:

In [11]:
print(f"Number of assay variables: {len(dataset.assay_variables)}")

for i, variable in enumerate(dataset.assay_variables):
    print(f"\nVariable {i + 1}:")
    print(f"  - Name: {variable.name}")
    print(f"  - Description: {variable.description}")
    print(f"  - Unit: {variable.unit}")
    print(f"  - Value: {variable.value}")

Number of assay variables: 2

Variable 1:
  - Name: PH
  - Description: pH level of the samples
  - Unit: pH
  - Value: None

Variable 2:
  - Name: T
  - Description: Temperature level of the samples
  - Unit: C
  - Value: None


## Accessing Structures

In [12]:
# Access the structures
structures = dataset.structures

# Obtain a specific structure:
my_structure = structures[0]

# We can get a summary of the metadata encoded in this assay:
for field in dataclasses.fields(my_structure):
    print(f"Found a field:\n{field}\n------------")

Found a field:
Field(name='name',type=<class 'str'>,default=<dataclasses._MISSING_TYPE object at 0x101ecdfd0>,default_factory=<dataclasses._MISSING_TYPE object at 0x101ecdfd0>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=False,doc=None,_field_type=_FIELD)
------------
Found a field:
Field(name='value',type=<class 'biotite.structure.AtomArray'>,default=<dataclasses._MISSING_TYPE object at 0x101ecdfd0>,default_factory=<dataclasses._MISSING_TYPE object at 0x101ecdfd0>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=False,doc=None,_field_type=_FIELD)
------------
Found a field:
Field(name='description',type=str | None,default=None,default_factory=<dataclasses._MISSING_TYPE object at 0x101ecdfd0>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=False,doc=None,_field_type=_FIELD)
------------
Found a field:
Field(name='metadata',type=dict[str, str],default=<dataclasses._MISSING_TYPE object at 0x101ecdfd0

When you access the structure, we return a Biotite `AtomArray` object. Biotite follows a vectorized approach where structure information is stored in NumPy arrays. You can access coordinates and annotations (like residue names, chain IDs, etc.) directly as arrays. See https://www.biotite-python.org/ for more information.

The following attributes can be used to extract specific atom or residue data:

```python
structure.coord      # NumPy array of atomic coordinates
structure.res_name   # Residue names (e.g., 'GLY')
structure.atom_name  # Atom names (e.g., 'CA')
structure.res_id     # Residue IDs
structure.chain_id   # Chain IDs
structure.element    # Chemical elements
```

In [13]:
biotite_structure = my_structure.value

print(f"Number of atoms: {biotite_structure.array_length()}")
print(f"First 5 residue names: {biotite_structure.res_name[:11]}")
print(f"First 5 atom names: {biotite_structure.atom_name[:11]}")
print(f"Coordinates of first atom: {biotite_structure.coord[0]}")

Number of atoms: 1198
First 5 residue names: ['PHE' 'PHE' 'PHE' 'PHE' 'PHE' 'PHE' 'PHE' 'PHE' 'PHE' 'PHE' 'PHE']
First 5 atom names: ['N' 'CA' 'C' 'CB' 'O' 'CG' 'CD1' 'CD2' 'CE1' 'CE2' 'CZ']
Coordinates of first atom: [-28.907  -2.327  45.829]


## Accessing MSAs (Multiple Sequence Alignments)

MSAs provide evolutionary information through aligned sequences. Similarly to structures we return the biopython object to access the msa data.

In [14]:
# Access the MSA data
msas = dataset.msas

my_msa = msas[0]
print(my_msa)

MSA(
	name='msa',
	description: None,
	value:
		tr|A0A1I9GEU1|A0A1I9GEU1_NEIME Pilin OS=Neisseria meningitidis OX=487 PE=1 SV=1 FTLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQKSAVTEY
		tr|A0A1D3IPW2|A0A1D3IPW2_NEIGO Pilin OS=Neisseria gonorrhoeae OX=485 GN=pilE_7 PE=4 SV=1 --------------MRQNirarpadlPVFECGRqrVFIHRPAAPAinqNR
		tr|E0NC80|E0NC80_NEIME Fimbrial family protein OS=Neisseria meningitidis ATCC 13091 OX=862513 GN=HMPREF0602_2112 PE=4 SV=1 --------------------------------------------------
		...
)


In [15]:
# A biotite MSA (biotite.sequence.io.fasta.FastaFile) object is a dictionary-like object:

biotite_msa = my_msa.value
print(biotite_msa)

>tr|A0A1I9GEU1|A0A1I9GEU1_NEIME Pilin OS=Neisseria meningitidis OX=487 PE=1 SV=1
FTLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQKSAVTEYYLNHGEWPGDNSSAGVATSADIKGKYVQSVTVANGVITAQMASSNVNNEIKSKKLSLWAKRQNGSVKWFCGQPVTRTTATATDVAAANGKTDDKINTKHLPSTCRDDSSAS
>tr|A0A1D3IPW2|A0A1D3IPW2_NEIGO Pilin OS=Neisseria gonorrhoeae OX=485 GN=pilE_7 PE=4 SV=1
--------------MRQNirarpadlPVFECGRqrVFIHRPAAPAinqNRTGLHLPQLVFAGHYPNNGKWPANNGNAGVASp-ADIKGKYVESVTVANGVVTAQMKPSGVNNEIKDKRLSLWGRRENGSVKWFCGQPVTRTKADAD-DV--KADGTKKIETKHLPSTCRDTSSA-
>tr|E0NC80|E0NC80_NEIME Fimbrial family protein OS=Neisseria meningitidis ATCC 13091 OX=862513 GN=HMPREF0602_2112 PE=4 SV=1
----------------------------------------------------------------MVEGqKSAVTEYYLNHGKWPGGNSDA---GVASSSEIKGKELSLWAKRQDGSVKWFCGQPVERNAKATADAv-TAATPDTDKINTKHLPSTCRDAASAV
>tr|A0A0M3GY31|A0A0M3GY31_NEIGO Uncharacterized protein (Fragment) OS=Neisseria gonorrhoeae MIA_2011_03-10 OX=1351790 GN=M736_11665 PE=4 SV=1
-------------------------------------------------------------

In [16]:
# Accessing individual sequences in the MSA:
header, sequence = list(biotite_msa.items())[0]
print(header, sequence[:50])

tr|A0A1I9GEU1|A0A1I9GEU1_NEIME Pilin OS=Neisseria meningitidis OX=487 PE=1 SV=1 FTLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQKSAVTEY


## Accessing Sequences
We also record the reference sequence(s) of the dataset in the `[[ sequence ]]` section. This is helpful for engineering compared to a `wild-type` or `starting_sequence` and `engineered_sequence` from previous campaigns.

**It is important to note that the sequences in dataset.sequences are your reference sequences. The mutated sequences belong to an assay are what you most likely use for your ML model. See 

In [17]:
# Access the list of reference sequences
sequences = dataset.sequences

# Obtain a specific sequence
my_sequence = sequences[0]

print(f"Sequence name: {my_sequence.name}")
print(f"Sequence description: {my_sequence.description}")
print(f"Sequence type: {my_sequence.type}")
print(f"Sequence: {my_sequence.value[0:20]}....")

Sequence name: tr|A0A1I9GEU1|A0A1I9GEU1_NEIME
Sequence description: tr|A0A1I9GEU1|A0A1I9GEU1_NEIME Pilin OS=Neisseria meningitidis OX=487 PE=1 SV=1
Sequence type: wild_type
Sequence: FTLIELMIVIAIVGILAAVA....


## Summary

In this notebook, we've learned how to:

1. **Load datasets** from manifests and archives
2. **Access different data types**: sequences, structures, MSAs, and assays
3. **Access metadata** and variables

The PG2 Dataset package provides a powerful and flexible way to work with protein data in machine learning workflows. The standardized API makes it easy to switch between different datasets while maintaining consistent code structure.

## Next Steps

Now you're ready to:
- Use PG2 Dataset in your own ML projects
- Share standardized datasets with collaborators
- Take a look at `04_Data_Access_for_ML` to get started connecting the data to your machine learning models.
